# v16.9 — sinus bradikardia dan detak lambat pada S1 (D-1 + episode AF ≥ 30 s)

Evaluasi saja, dari cache v16.2–v16.7 (tanpa data baru, tanpa training). Rancangan dikunci sebelum alarm per ritme/pita dihitung:
[RANCANGAN.md](RANCANGAN.md) (sha256 di `../logs/v16.9_praregistrasi.log`). Logika di `bradikardia.py`.

- **S1** = D-1 (ECGFounder zero-shot, d1 > −1) + aturan episode AF ≥ 30 s (v16.5).
- **R** = paket v3.1 + aturan yang sama (referensi).

In [1]:
import time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import bradikardia as B
import sys; sys.path[:0] = ["../v11_model_tim"]
from bersama import evaluasi as Ev
t0 = time.time()
R = {}
D, A, P = {}, {}, {}
for ds in B.DATASET:
    d = B.muat(ds)
    al = B.sistem(d)
    D[ds], A[ds] = d, al
    P[ds] = {}
    for s, a in al.items():
        pl, ref, ref_idx = B.episode(d, a)
        B.cek_paritas(d, a, pl)
        P[ds][s] = pl
    P[ds]["ref"], P[ds]["ref_idx"] = ref, ref_idx
R["data"] = {ds: dict(window=int(d["rec"].size), unit=int(np.unique(d["unit"]).size), jam=B.jam(d["rec"].size),
                      jam_non_af=B.jam(int((d["y2"] == 0).sum())), jam_lambat=B.jam(int(((d["y2"] == 0) & (d["hr"] < 60)).sum())))
             for ds, d in D.items()}
display(pd.DataFrame(R["data"]).T.round(1))
print("paritas episode palsu dengan episode.metrik: OK")

,window,unit,jam,jam_non_af,jam_lambat
LTAFDB,1272328.0,84.0,1767.1,785.8,173.3
MIT-BIH,14152.0,46.0,19.7,17.6,4.3
CPSC 2021,321818.0,105.0,447.0,292.2,80.8
NSRDB,294162.0,18.0,408.6,408.6,52.9
EDB,126578.0,90.0,175.8,175.5,48.3
SHDB-AF,1659539.0,93.0,2304.9,1828.1,504.9
afpdb,35465.0,100.0,49.3,49.3,7.6
incartdb,25150.0,32.0,34.9,33.9,9.8
svdb,27812.0,78.0,38.6,38.6,5.8
Icentia11k,384436.0,600.0,533.9,315.9,94.0


paritas episode palsu dengan episode.metrik: OK


## 1. P1: sinus bradikardia berlabel (LTAFDB)

In [2]:
d = D["LTAFDB"]
rit = {s: B.ritme_episode(d, P["LTAFDB"][s]) for s in ("S1", "R")}
p1 = {}
for ritme in ("(SBR", "(N"):
    us1, ur = B.laju_ritme(d, P["LTAFDB"]["S1"], rit["S1"], ritme), B.laju_ritme(d, P["LTAFDB"]["R"], rit["R"], ritme)
    den = (d["ritme"] == ritme) & (d["y2"] == 0)
    b = B.boot(us1, ur)
    fs1 = B.boot(B.per_unit(d["unit"], A["LTAFDB"]["S1"] & den, den), B.per_unit(d["unit"], A["LTAFDB"]["R"] & den, den))
    p1[ritme] = dict(jam=B.jam(int(den.sum())), subjek=b["n_unit"], palsu_s1=sum(v[0] for v in us1.values()),
                     palsu_r=sum(v[0] for v in ur.values()), laju=b, fraksi_waktu=fs1)
    print(f"{ritme}: {p1[ritme]['jam']:.1f} jam, {b['n_unit']} subjek | episode palsu S1 {p1[ritme]['palsu_s1']:.0f}, R {p1[ritme]['palsu_r']:.0f}"
          f" | laju S1 {b['nilai']:.2f}/jam [{b['ci'][0]:.2f}–{b['ci'][1]:.2f}] vs R {b['nilai_b']:.2f} [{b['ci_b'][0]:.2f}–{b['ci_b'][1]:.2f}],"
          f" P(Δ>0) = {b['p_lebih']:.3f} | waktu dialarmkan S1 {fs1['nilai']*100:.1f}% vs R {fs1['nilai_b']*100:.1f}%")
# ritme semua episode palsu (deskriptif)
R["ritme_episode_palsu"] = {s: pd.Series(rit[s]).value_counts().to_dict() for s in rit}
print(R["ritme_episode_palsu"])
sb = p1["(SBR"]
R["P1"] = p1
R["P1_kelemahan_nyata"] = bool(sb["laju"]["nilai"] > 1.0 and sb["laju"]["p_lebih"] >= 0.95)
print("P1 kelemahan nyata S1:", R["P1_kelemahan_nyata"])

(SBR: 35.4 jam, 25 subjek | episode palsu S1 315, R 17 | laju S1 8.89/jam [2.32–14.93] vs R 0.48 [0.00–2.15], P(Δ>0) = 0.998 | waktu dialarmkan S1 12.1% vs R 0.5%


(N: 746.6 jam, 50 subjek | episode palsu S1 495, R 1612 | laju S1 0.66/jam [0.12–1.57] vs R 2.16 [1.04–3.60], P(Δ>0) = 0.000 | waktu dialarmkan S1 1.0% vs R 2.8%
{'S1': {'(N': 495, '(SBR': 315, '(SVTA': 3, '(T': 2, '(AB': 1}, 'R': {'(N': 1612, '(AB': 24, '(SBR': 17, '(SVTA': 5, '(B': 3, '(T': 3}}
P1 kelemahan nyata S1: True


## 2. P2: per subjek dan jarak ke episode AF

In [3]:
us1 = B.laju_ritme(d, P["LTAFDB"]["S1"], rit["S1"], "(SBR")
ur = B.laju_ritme(d, P["LTAFDB"]["R"], rit["R"], "(SBR")
tab = pd.DataFrame({u: dict(jam_sbr=v[1], palsu_s1=v[0], palsu_r=ur.get(u, [0, 0])[0]) for u, v in us1.items()}).T
tab = tab.sort_values("palsu_s1", ascending=False)
display(tab.round(2))
tot = tab["palsu_s1"].sum()
top3 = tab["palsu_s1"].head(3).sum() / tot if tot else float("nan")
jarak = {}
for s in ("S1", "R"):
    jarak[s] = [B.jarak_ke_af(e["a"], e["b"], P["LTAFDB"]["ref"][e["rec"]]) for e, r in zip(P["LTAFDB"][s], rit[s]) if r == "(SBR"]
msb = np.flatnonzero((d["ritme"] == "(SBR") & (d["y2"] == 0))
dasar = [B.jarak_ke_af(d["start"][i], d["start"][i] + B.W, P["LTAFDB"]["ref"][d["rec"][i]]) for i in msb]
R["P2"] = dict(per_subjek=tab.to_dict("index"), porsi_top3_s1=top3, subjek_dengan_palsu_s1=int((tab["palsu_s1"] > 0).sum()),
               jarak_s1=B.kelas_jarak(jarak["S1"]), jarak_r=B.kelas_jarak(jarak["R"]), jarak_dasar=B.kelas_jarak(dasar),
               median_jarak_s1=float(np.median(jarak["S1"])) if jarak["S1"] else float("nan"))
print(f"porsi 3 subjek teratas: {top3:.2f}; subjek dengan episode palsu S1: {R['P2']['subjek_dengan_palsu_s1']} dari {len(tab)}")
display(pd.DataFrame({"S1 palsu": R["P2"]["jarak_s1"], "R palsu": R["P2"]["jarak_r"], "semua window SBR": R["P2"]["jarak_dasar"]}).round(3))

,jam_sbr,palsu_s1,palsu_r
53,14.13,146.0,2.0
13,4.40,111.0,0.0
37,1.94,27.0,0.0
47,1.03,12.0,14.0
03,4.99,11.0,0.0
74,0.16,5.0,1.0
35,1.01,3.0,0.0
105,1.04,0.0,0.0
01,0.03,0.0,0.0
15,0.05,0.0,0.0


porsi 3 subjek teratas: 0.90; subjek dengan episode palsu S1: 7 dari 25


,S1 palsu,R palsu,semua window SBR
< 10 menit,0.270,0.118,0.136
10–60 menit,0.314,0.059,0.344
> 60 menit,0.416,0.824,0.520
rekaman tanpa AF,0.000,0.000,0.000


## 3. P3: detak lambat lintas dataset (window non-AF)

In [4]:
p3, per_pita = {}, {}
for ds in B.DATASET:
    d, al = D[ds], A[ds]
    o = {}
    for s in ("S1", "R"):
        lam = B.fraksi_pita(d, al[s], *B.LAMBAT)
        acu = B.fraksi_pita(d, al[s], *B.ACUAN)
        b = B.boot(lam, acu)
        o[s] = dict(lambat=b["nilai"], acuan=b["nilai_b"], ci_lambat=b["ci"], ci_acuan=b["ci_b"], delta=b["delta"], ci_delta=b["ci_delta"],
                    p_lebih=b["p_lebih"])
    o["S1_lemah"] = bool(o["S1"]["p_lebih"] >= 0.95 and o["S1"]["lambat"] >= 2 * o["S1"]["acuan"])
    p3[ds] = o
    per_pita[ds] = {}
    for lo, hi, nama in B.PITA:
        den = (d["y2"] == 0) & B.pita(d["hr"], lo, hi)
        per_pita[ds][nama] = dict(jam=B.jam(int(den.sum())),
                                  **{f"fraksi_{s}": float((al[s] & den).sum() / den.sum()) if den.sum() else float("nan") for s in ("S1", "R")},
                                  **{f"laju_{s}": B.boot(B.laju_pita(d, P[ds][s], lo, hi))["nilai"] for s in ("S1", "R")})
tabel = pd.DataFrame({ds: {"S1 lambat %": o["S1"]["lambat"] * 100, "S1 acuan %": o["S1"]["acuan"] * 100, "S1 P(Δ>0)": o["S1"]["p_lebih"],
                           "R lambat %": o["R"]["lambat"] * 100, "R acuan %": o["R"]["acuan"] * 100, "R P(Δ>0)": o["R"]["p_lebih"],
                           "S1 lemah": o["S1_lemah"]} for ds, o in p3.items()}).T
display(tabel)
lain = [ds for ds in B.DATASET if ds != "LTAFDB" and p3[ds]["S1_lemah"]]
R["P3"], R["per_pita"] = p3, per_pita
R["P3_dataset_lemah_selain_ltafdb"] = lain
R["P3_kelemahan_umum"] = len(lain) >= 2
print("dataset selain LTAFDB dengan S1 lemah di detak lambat:", lain, "→ kelemahan umum:", R["P3_kelemahan_umum"])
for ds in B.DATASET:
    print(ds, {k: f"{v['jam']:.0f} jam S1 {v['fraksi_S1']*100:.2f}% / R {v['fraksi_R']*100:.2f}%; palsu/jam S1 {v['laju_S1']:.2f} R {v['laju_R']:.2f}"
               for k, v in per_pita[ds].items()})

,S1 lambat %,S1 acuan %,S1 P(Δ>0),R lambat %,R acuan %,R P(Δ>0),S1 lemah
LTAFDB,5.070768,0.453837,0.997,2.813086,1.781399,0.719,True
MIT-BIH,0.16239,0.105312,0.54,3.475154,3.826352,0.468,False
CPSC 2021,3.293249,1.600423,0.7845,8.534616,5.105,0.816,False
NSRDB,1.48654,0.822125,0.79,6.581747,2.886685,0.9805,False
EDB,0.491973,0.030282,0.997,0.808447,0.443325,0.8545,True
SHDB-AF,3.11115,1.340253,0.876,3.479239,2.927476,0.665,False
afpdb,2.888483,0.151204,0.8705,2.084095,1.702012,0.72,False
incartdb,0.26821,0.895232,0.2215,2.724449,6.876419,0.005,False
svdb,3.194811,3.407865,0.466,1.032909,7.605178,0.0005,False
Icentia11k,1.693237,0.095187,1.0,3.24611,2.594812,0.7245,True


dataset selain LTAFDB dengan S1 lemah di detak lambat: ['EDB', 'Icentia11k'] → kelemahan umum: True
LTAFDB {'< 50': '40 jam S1 11.72% / R 2.19%; palsu/jam S1 8.22 R 1.61', '50–60': '134 jam S1 3.10% / R 3.00%; palsu/jam S1 1.86 R 2.02', '60–80': '433 jam S1 0.54% / R 1.58%; palsu/jam S1 0.36 R 1.46', '80–100': '156 jam S1 0.22% / R 2.33%; palsu/jam S1 0.31 R 2.71', '≥ 100': '23 jam S1 1.49% / R 25.19%; palsu/jam S1 1.63 R 11.69'}
MIT-BIH {'< 50': '1 jam S1 0.00% / R 6.48%; palsu/jam S1 0.00 R 5.71', '50–60': '3 jam S1 0.22% / R 2.50%; palsu/jam S1 0.31 R 2.48', '60–80': '7 jam S1 0.00% / R 2.03%; palsu/jam S1 0.00 R 1.77', '80–100': '5 jam S1 0.25% / R 6.22%; palsu/jam S1 0.20 R 4.32', '≥ 100': '1 jam S1 0.89% / R 28.83%; palsu/jam S1 1.42 R 17.77'}
CPSC 2021 {'< 50': '27 jam S1 6.04% / R 10.11%; palsu/jam S1 3.73 R 6.82', '50–60': '54 jam S1 1.93% / R 7.75%; palsu/jam S1 1.68 R 5.04', '60–80': '151 jam S1 1.68% / R 5.11%; palsu/jam S1 0.71 R 4.02', '80–100': '51 jam S1 1.37% / R 5.08%

## 4. P4: AF berdetak lambat (deskriptif)

In [5]:
p4 = {}
for ds in B.DATASET:
    if D[ds]["y2"].sum() == 0:
        continue
    p4[ds] = B.af_lambat(D[ds], A[ds]["S1"], P[ds]["ref_idx"])
R["P4"] = p4
display(pd.DataFrame(p4).T)

,window_af,frac_window_lambat,episode_30,episode_lambat,frac_episode_lambat,se_s1_lambat
LTAFDB,706582.0,0.108480,4347.0,1211.0,0.278583,0.968621
MIT-BIH,1514.0,0.009247,44.0,0.0,0.000000,NaN
CPSC 2021,111464.0,0.151654,1828.0,205.0,0.112144,0.873171
EDB,247.0,0.000000,1.0,0.0,0.000000,NaN
SHDB-AF,343286.0,0.117709,1648.0,101.0,0.061286,1.000000
incartdb,739.0,0.008119,4.0,0.0,0.000000,NaN
Icentia11k,156995.0,0.160336,3975.0,518.0,0.130314,0.893822


## 5. Eksplorasi setelah hasil (tidak dikunci)

Mengapa S1 mengalarmkan sinus bradikardia LTAFDB? Untuk setiap window `(SBR`, detak dibaca dari **anotasi referensi** (.atr, bukan
detektor perangkat): jumlah detak, detak (bpm), CV RR, dan fraksi detak bukan normal. Dibandingkan antara window yang berada di dalam
episode alarm S1 dan yang tidak. Ini deskriptif dan dilihat setelah hasil P1–P4; tidak dipakai memilih apa pun.

In [6]:
d = D["LTAFDB"]
msb = np.flatnonzero((d["ritme"] == "(SBR") & (d["y2"] == 0))
ref = np.full((msb.size, 4), np.nan)
for r in np.unique(d["rec"][msb]):
    k = msb[d["rec"][msb] == r]
    ref[np.searchsorted(msb, k)] = B.detak_referensi(r, d["start"][k])
al = A["LTAFDB"]["S1"][msb]
eks = {}
for nama, m in (("dialarmkan S1", al), ("tidak dialarmkan", ~al)):
    x = ref[m]
    eks[nama] = dict(window=int(m.sum()), tanpa_anotasi=float(np.mean(np.isnan(x[:, 0]))),
                     hr_detektor_median=float(np.median(d["hr"][msb][m])), hr_ref_median=float(np.nanmedian(x[:, 1])),
                     cv_ref_median=float(np.nanmedian(x[:, 2])), frac_cv_ref_gt_010=float(np.nanmean(x[:, 2] > 0.10)),
                     ektopik_median=float(np.nanmedian(x[:, 3])), frac_ada_ektopik=float(np.nanmean(x[:, 3] > 0)),
                     selisih_hr_median=float(np.nanmedian(d["hr"][msb][m] - x[:, 1])),
                     frac_hr_ref_lt_45=float(np.nanmean(x[:, 1] < 45)))
display(pd.DataFrame(eks).round(3))
# per subjek: tiga subjek teratas vs lainnya
top = list(R["P2"]["per_subjek"])[:3]
ps = {}
for u in np.unique(d["unit"][msb]):
    m = d["unit"][msb] == u
    ps[str(u)] = dict(window=int(m.sum()), alarm=float(al[m].mean()), hr_ref=float(np.nanmedian(ref[m, 1])), cv_ref=float(np.nanmedian(ref[m, 2])),
                      ektopik=float(np.nanmean(ref[m, 3] > 0)))
tps = pd.DataFrame(ps).T.sort_values("alarm", ascending=False)
display(tps.head(10).round(3))
# pita detak referensi: fraksi dialarmkan S1
pr = {}
for lo, hi, nama in ((0, 40, "< 40"), (40, 45, "40–45"), (45, 50, "45–50"), (50, 55, "50–55"), (55, 60, "55–60"), (60, np.inf, "≥ 60")):
    m = (ref[:, 1] >= lo) & (ref[:, 1] < hi)
    pr[nama] = dict(window=int(m.sum()), alarm_s1=float(al[m].mean()) if m.sum() else float("nan"),
                    alarm_r=float(A["LTAFDB"]["R"][msb][m].mean()) if m.sum() else float("nan"))
display(pd.DataFrame(pr).T.round(3))
R["eksplorasi"] = dict(alarm_vs_tidak=eks, per_subjek=tps.to_dict("index"), pita_hr_ref=pr, top3=top)

,dialarmkan S1,tidak dialarmkan
window,3073.000,22428.000
tanpa_anotasi,0.000,0.000
hr_detektor_median,44.908,45.415
hr_ref_median,44.956,45.521
cv_ref_median,0.019,0.019
frac_cv_ref_gt_010,0.084,0.095
ektopik_median,0.000,0.000
frac_ada_ektopik,0.063,0.084
selisih_hr_median,-0.011,-0.019
frac_hr_ref_lt_45,0.508,0.475


,window,alarm,hr_ref,cv_ref,ektopik
13,3167.0,0.377,46.305,0.010,0.087
74,114.0,0.237,38.847,0.180,0.693
53,10172.0,0.148,35.101,0.024,0.061
37,1399.0,0.133,47.282,0.040,0.300
47,739.0,0.108,37.030,0.018,0.129
35,726.0,0.021,46.697,0.025,0.041
03,3593.0,0.018,46.952,0.014,0.025
105,749.0,0.000,48.696,0.009,0.011
01,19.0,0.000,49.548,0.119,0.632
15,39.0,0.000,40.996,0.153,0.615


,window,alarm_s1,alarm_r
< 40,9183.0,0.032,0.008
40–45,3031.0,0.419,0.015
45–50,12832.0,0.117,0.000
50–55,443.0,0.016,0.000
55–60,12.0,0.000,0.000
≥ 60,0.0,NaN,NaN


## 6. Kesimpulan

In [7]:
sb = R["P1"]["(SBR"]
print(f"P1 LTAFDB sinus bradikardia ({sb['jam']:.1f} jam, {sb['subjek']} subjek): S1 {sb['laju']['nilai']:.2f} palsu/jam "
      f"[{sb['laju']['ci'][0]:.2f}–{sb['laju']['ci'][1]:.2f}] vs R {sb['laju']['nilai_b']:.2f}; P(Δ>0) = {sb['laju']['p_lebih']:.3f}; "
      f"waktu dialarmkan {sb['fraksi_waktu']['nilai']*100:.1f}% vs {sb['fraksi_waktu']['nilai_b']*100:.1f}% → kelemahan nyata: {R['P1_kelemahan_nyata']}")
n = R["P1"]["(N"]
print(f"   pembanding sinus normal LTAFDB: S1 {n['laju']['nilai']:.2f} palsu/jam vs R {n['laju']['nilai_b']:.2f}")
print(f"P2 porsi 3 subjek teratas {R['P2']['porsi_top3_s1']:.2f}; jarak ke AF S1: {R['P2']['jarak_s1']}")
print(f"P3 kelemahan umum detak lambat: {R['P3_kelemahan_umum']} (dataset lain yang lemah: {R['P3_dataset_lemah_selain_ltafdb']})")
for ds, v in R["P4"].items():
    print(f"P4 {ds}: window AF < 60 bpm {v['frac_window_lambat']*100:.1f}%, episode lambat {v['episode_lambat']}/{v['episode_30']}, Se S1 {v['se_s1_lambat']}")
R["lingkungan"] = Ev._lingkungan(); R["durasi_menit"] = (time.time() - t0) / 60
B.simpan_json(R, B.CACHE / "ringkasan.json"); print("tersimpan")

P1 LTAFDB sinus bradikardia (35.4 jam, 25 subjek): S1 8.89 palsu/jam [2.32–14.93] vs R 0.48; P(Δ>0) = 0.998; waktu dialarmkan 12.1% vs 0.5% → kelemahan nyata: True
   pembanding sinus normal LTAFDB: S1 0.66 palsu/jam vs R 2.16
P2 porsi 3 subjek teratas 0.90; jarak ke AF S1: {'< 10 menit': 0.2698412698412698, '10–60 menit': 0.3142857142857143, '> 60 menit': 0.4158730158730159, 'rekaman tanpa AF': 0.0}
P3 kelemahan umum detak lambat: True (dataset lain yang lemah: ['EDB', 'Icentia11k'])
P4 LTAFDB: window AF < 60 bpm 10.8%, episode lambat 1211/4347, Se S1 0.9686209744013212
P4 MIT-BIH: window AF < 60 bpm 0.9%, episode lambat 0/44, Se S1 nan
P4 CPSC 2021: window AF < 60 bpm 15.2%, episode lambat 205/1828, Se S1 0.8731707317073171
P4 EDB: window AF < 60 bpm 0.0%, episode lambat 0/1, Se S1 nan
P4 SHDB-AF: window AF < 60 bpm 11.8%, episode lambat 101/1648, Se S1 1.0
P4 incartdb: window AF < 60 bpm 0.8%, episode lambat 0/4, Se S1 nan
P4 Icentia11k: window AF < 60 bpm 16.0%, episode lambat 518/